In [ ]:
import os
import pandas as pd
import psycopg2
from psycopg2 import sql

# ---------- CONFIGURATION ----------
CSV_FOLDER = r"C:\Users\leedf\weatherData"  # Folder containing the CSV files
DB_NAME = "weather"
DB_USER = "postgres"
DB_PASSWORD = "071726postGRES"
DB_HOST = "localhost"
DB_PORT = "5432"

# ---------- CONNECT TO POSTGRES ----------
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cursor = conn.cursor()
    print("Connected to PostgreSQL.")
except Exception as e:
    raise SystemExit(f"Database connection failed: {e}")

# ---------- PROCESS EACH CSV ----------
for file in os.listdir(CSV_FOLDER):
    if file.endswith(".csv"):
        table_name = os.path.splitext(file)[0]  # Table name from file name
        file_path = os.path.join(CSV_FOLDER, file)

        # Load CSV into DataFrame
        df = pd.read_csv(file_path)

        # Validate required columns
        required_cols = {"zip", "state", "city"}
        if not required_cols.issubset(df.columns):
            print(f"Skipping {file}: Missing required columns.")
            continue

        # Create table SQL dynamically
        col_defs = []
        for col in df.columns:
            if col == "zip":
                col_defs.append(sql.SQL("{} INTEGER PRIMARY KEY").format(sql.Identifier(col)))
            else:
                col_defs.append(sql.SQL("{} TEXT").format(sql.Identifier(col)))

        create_table_query = sql.SQL("CREATE TABLE IF NOT EXISTS {} ({});").format(
            sql.Identifier(table_name),
            sql.SQL(", ").join(col_defs)
        )

        try:
            cursor.execute(create_table_query)
            print(f"Table '{table_name}' created.")
        except Exception as e:
            print(f"Error creating table {table_name}: {e}")
            continue

        # Insert data
        for _, row in df.iterrows():
            insert_query = sql.SQL("INSERT INTO {} ({}) VALUES ({}) ON CONFLICT (zip) DO NOTHING;").format(
                sql.Identifier(table_name),
                sql.SQL(", ").join(map(sql.Identifier, df.columns)),
                sql.SQL(", ").join(sql.Placeholder() * len(df.columns))
            )
            try:
                cursor.execute(insert_query, tuple(row))
            except Exception as e:
                print(f"Insert error in {table_name}: {e}")

print("All CSV files processed.")

# ---------- CLEANUP ----------
cursor.close()
conn.close()


Connected to PostgreSQL.
Table 'us_average_humidity_by_zip' created.
Table 'us_average_precipitation_by_zip' created.
Table 'us_average_snowfall_by_zip' created.
Table 'us_average_temperature_by_zip' created.
All CSV files processed.


In [ ]:
# Bring in the wind data to use as a PostgreSql table
import pandas as pd
wind_data = pd.read_csv(r"C:\Users\leedf\Downloads\annual_avg_windspeed_data.csv")
wind_data.head()


,zip_code,latitude,longitude,station_id,annual_avg_awnd_mph
0,98253,48.098624,-122.580049,GHCND:USW00024255,No AWND data for year
1,98271,48.118345,-122.171071,GHCND:USW00024255,No AWND data for year
2,98324,48.091757,-123.172486,GHCND:USW00094276,4.841630901287553
3,99148,48.082743,-117.620107,GHCND:USW00024157,7.996952908587265
4,98834,48.140121,-120.019759,GHCND:USC00451350,2.0835654596100213


In [5]:
import os
import pandas as pd
import psycopg2
from psycopg2 import sql

# ---------- CONFIGURATION ----------
CSV_FOLDER = r"C:\Users\leedf\wind_speed"  # Folder containing the CSV files
DB_NAME = "weather"
DB_USER = "postgres"
DB_PASSWORD = "071726postGRES"
DB_HOST = "localhost"
DB_PORT = "5432"

# ---------- CONNECT TO POSTGRES ----------
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cursor = conn.cursor()
    print("Connected to PostgreSQL.")
except Exception as e:
    raise SystemExit(f"Database connection failed: {e}")

# ---------- PROCESS EACH CSV ----------
for file in os.listdir(CSV_FOLDER):
    if file.endswith(".csv"):
        table_name = os.path.splitext(file)[0]  # Table name from file name
        file_path = os.path.join(CSV_FOLDER, file)

        # Load CSV into DataFrame
        df = pd.read_csv(file_path)

        # Validate required columns
        required_cols = {"zip_code", "latitude", "longitude", "station_id"}
        if not required_cols.issubset(df.columns):
            print(f"Skipping {file}: Missing required columns.")
            continue

        # Create table SQL dynamically
        col_defs = []
        for col in df.columns:
            if col == "zip_code":
                col_defs.append(sql.SQL("{} INTEGER PRIMARY KEY").format(sql.Identifier(col)))
            else:
                col_defs.append(sql.SQL("{} TEXT").format(sql.Identifier(col)))

        create_table_query = sql.SQL("CREATE TABLE IF NOT EXISTS {} ({});").format(
            sql.Identifier(table_name),
            sql.SQL(", ").join(col_defs)
        )

        try:
            cursor.execute(create_table_query)
            print(f"Table '{table_name}' created.")
        except Exception as e:
            print(f"Error creating table {table_name}: {e}")
            continue

        # Insert data
        for _, row in df.iterrows():
            insert_query = sql.SQL("INSERT INTO {} ({}) VALUES ({}) ON CONFLICT (zip_code) DO NOTHING;").format(
                sql.Identifier(table_name),
                sql.SQL(", ").join(map(sql.Identifier, df.columns)),
                sql.SQL(", ").join(sql.Placeholder() * len(df.columns))
            )
            try:
                cursor.execute(insert_query, tuple(row))
            except Exception as e:
                print(f"Insert error in {table_name}: {e}")

print("All CSV files processed.")

# ---------- CLEANUP ----------
cursor.close()
conn.close()


Connected to PostgreSQL.
Table 'annual_avg_windspeed_data' created.
All CSV files processed.


In [9]:
# Create a pandas dataframe from merged tables in a PostgreSQL database
import pandas as pd
from sqlalchemy import create_engine

# ---------- CONFIGURATION ----------
DB_NAME = "weather"
DB_USER = "postgres"
DB_PASSWORD = "071726postGRES"
DB_HOST = "localhost"
DB_PORT = "5432"

# ---------- CONNECT TO POSTGRES ----------
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# ---------- ALTER TABLES DATATYPES ----------
query = '''
SELECT 
    h.*, 
    p.avg_monthly_precipitation_in AS precipitation, 
    s.avg_monthly_snowfall_in AS snowfall, 
    t.avg_temperature_f AS temperature , 
    w.annual_avg_awnd_mph AS wind 
FROM us_average_humidity_by_zip AS h 
INNER JOIN us_average_precipitation_by_zip AS p ON h.zip = p.zip
INNER JOIN us_average_snowfall_by_zip AS s ON h.zip = s.zip
INNER JOIN us_average_temperature_by_zip AS t ON h.zip = t.zip
INNER JOIN annual_avg_windspeed_data AS w ON h.zip = w.zip_code;
'''

# Read the joined query into a pandas dataframe
df = pd.read_sql(query, con=engine)

# Display the first few rows of the dataframe
df.head()

,zip,city,state,avg_relative_humidity_pct,precipitation,snowfall,temperature,wind
0,98253,Greenbank,WA,75.294738705695,2.50803943624694,0.3562992125984252,51.58910360940877,No AWND data for year
1,98271,Marysville,WA,78.26610605831172,2.50803943624694,0.3562992125984252,50.7394606643558,No AWND data for year
2,98324,Carlsborg,WA,78.83734773630276,2.275196850393701,0.7027559055118111,49.64409138071476,4.841630901287553
3,99148,Loon Lake,WA,66.87719089337939,1.7293307086614174,3.762139107611549,47.096120338678034,7.996952908587265
4,98834,Methow,WA,59.05169803067157,1.2137795275590553,3.768747656542932,51.04268979565974,2.0835654596100213


In [ ]:
# Save the dataframe to a CSV file
df.to_csv(r"C:\Users\leedf\weatherData\weather_data_v1.csv", index=False)

In [ ]:
#------------ Dataset Creation ------------ 

# Combine two dataframes on the state columns to create a new dataframe with the combined data

df1 = pd.read_csv(r"C:\Users\leedf\weatherData\weather_data_v1.csv")
df2 = pd.read_csv(r"C:\Users\leedf\weatherData\PeakSunHours.csv")
finalWeatherData = pd.merge(df1, df2, on='state', how='left')
finalWeatherData.to_csv(r"C:\Users\leedf\weatherData\finalWeatherData.csv", index=False)
finalWeatherData.head()


,zip,city,state,avg_relative_humidity_pct,precipitation,snowfall,temperature,wind,Peak sun hours/day
0,98253,Greenbank,WA,75.294739,2.508039,0.356299,51.589104,No AWND data for year,3.8
1,98271,Marysville,WA,78.266106,2.508039,0.356299,50.739461,No AWND data for year,3.8
2,98324,Carlsborg,WA,78.837348,2.275197,0.702756,49.644091,4.841630901287553,3.8
3,99148,Loon Lake,WA,66.877191,1.729331,3.762139,47.096120,7.996952908587265,3.8
4,98834,Methow,WA,59.051698,1.213780,3.768748,51.042690,2.0835654596100213,3.8


In [5]:
import pandas as pd
df = pd.read_csv(r"C:\Users\leedf\weatherData\finalWeatherData.csv")


In [ ]:
#------------- Clean the dataframe for better usability -------------

# Round the numeric columns to the appropriate decimal places depending on the measurement.
df[['snowfall','temperature','wind']]=df[['snowfall','temperature','wind']].apply(pd.to_numeric, errors='coerce').round(1)
df[['precipitation']]=df[['precipitation']].apply(pd.to_numeric, errors='coerce').round(2)
df[['avg_relative_humidity_pct']]=df[['avg_relative_humidity_pct']].apply(pd.to_numeric, errors='coerce').round()

# Bring in the peak sun hours data from the second dataframe
psh_data = pd.read_csv(r"C:\Users\leedf\weatherData\PeakSunHours.csv")

# Merge the peak sun hours data with the main dataframe on the 'state' column
fullWeatherData = pd.merge(df, psh_data, on='state', how='left')
#fullWeatherData.to_csv(r"C:\Users\leedf\weatherData\fullWeatherData.csv", index=False)

# Transform the 'snowfall' and 'precipitation' columns into annual averages by multiplying by 12 and round them to 2 decimal places
fullWeatherData['annual_avg_snowfall'] = (fullWeatherData['snowfall'] * 12)
fullWeatherData['annual_avg_precipitation'] = (fullWeatherData['precipitation'] * 12)
fullWeatherData[['annual_avg_snowfall','annual_avg_precipitation']] = fullWeatherData[['annual_avg_snowfall','annual_avg_precipitation']].round(2)

# Save the cleaned and transformed dataframe to a new CSV file
fullWeatherData.to_csv(r"C:\Users\leedf\weatherData\cleanedWeatherDataByZip.csv", index=False)


In [ ]:
#------------ Continue Cleaning the dataframe for better usability -------------

# Use KNN to fill in the 8948 missing values in the 'wind' column of the cleaned weather data by zip code and their related coordinates

# Read in the coordinates data for the zip codes
coord_df = pd.read_csv(r"C:\Users\leedf\weatherData\zipcode_lat_long.csv")

# Merge the df with the coordinates dataframe on the 'zip' column
mergedData = pd.merge(df, coord_df, on='zip', how='left')

# Create a KNN Regressor to fill in missing values in the 'wind' column based on the nearest neighbors in the dataset
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

# Split the data into complete(train) and incomplete(test) wind data
train_data = mergedData[mergedData['wind'].notna()]
test_data = mergedData[mergedData['wind'].isna()]

# Extract the Latitude and Longitude as features, with the 'wind' column as the target variable
X_train = train_data[['lat', 'lon']]
y_train = train_data['wind']
X_test = test_data[['lat', 'lon']]

# Fit KNN Regressor(using 7 neighbors) to the training data weighted by distance
knn = KNeighborsRegressor(n_neighbors=7, weights='distance')
knn.fit(X_train, y_train)

# Predict and fill in the missing 'wind' values in the test data
fullWeatherData.loc[fullWeatherData['wind'].isna(), 'wind'] = knn.predict(X_test)
fullWeatherData.head()

,zip,city,state,avg_relative_humidity_pct,precipitation,snowfall,temperature,wind,Peak sun hours/day_x,Peak sun hours/day_y,annual_avg_snowfall,annual_avg_precipitation
0,98253,Greenbank,WA,75.0,2.51,0.4,51.6,5.061560,3.8,3.8,4.8,30.12
1,98271,Marysville,WA,78.0,2.51,0.4,50.7,6.356402,3.8,3.8,4.8,30.12
2,98324,Carlsborg,WA,79.0,2.28,0.7,49.6,4.800000,3.8,3.8,8.4,27.36
3,99148,Loon Lake,WA,67.0,1.73,3.8,47.1,8.000000,3.8,3.8,45.6,20.76
4,98834,Methow,WA,59.0,1.21,3.8,51.0,2.100000,3.8,3.8,45.6,14.52


In [ ]:
#------------ Continue Cleaning the dataframe for better usability -------------

# Use KNN to fill the 623 missing values in the 'avg_relative_humidity_pct' column of the cleaned weather data by zip code and their related coordinates
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

# Split the data into complete(train) and incomplete(test) wind data
train_data = mergedData[mergedData['avg_relative_humidity_pct'].notna()]
test_data = mergedData[mergedData['avg_relative_humidity_pct'].isna()]

# Extract the Latitude and Longitude as features, with the 'avg_relative_humidity_pct' column as the target variable
X_train = train_data[['lat', 'lon']]
y_train = train_data['avg_relative_humidity_pct']
X_test = test_data[['lat', 'lon']]

# Fit KNN Regressor(using 7 neighbors) to the training data weighted by distance
knn = KNeighborsRegressor(n_neighbors=7, weights='distance')
knn.fit(X_train, y_train)

# Predict and fill in the missing 'avg_relative_humidity_pct' values in the test data
fullWeatherData.loc[fullWeatherData['avg_relative_humidity_pct'].isna(), 'avg_relative_humidity_pct'] = knn.predict(X_test)
fullWeatherData.head()

,zip,city,state,avg_relative_humidity_pct,precipitation,snowfall,temperature,wind,Peak sun hours/day_x,Peak sun hours/day_y,annual_avg_snowfall,annual_avg_precipitation
0,98253,Greenbank,WA,75.0,2.51,0.4,51.6,5.061560,3.8,3.8,4.8,30.12
1,98271,Marysville,WA,78.0,2.51,0.4,50.7,6.356402,3.8,3.8,4.8,30.12
2,98324,Carlsborg,WA,79.0,2.28,0.7,49.6,4.800000,3.8,3.8,8.4,27.36
3,99148,Loon Lake,WA,67.0,1.73,3.8,47.1,8.000000,3.8,3.8,45.6,20.76
4,98834,Methow,WA,59.0,1.21,3.8,51.0,2.100000,3.8,3.8,45.6,14.52


In [ ]:
#------------ Continue Cleaning the dataframe for better usability -------------

# Format the columns that were imputed by KNN to the appropriate decimal places depending on the measurement.
fullWeatherData[['avg_relative_humidity_pct']]=fullWeatherData[['avg_relative_humidity_pct']].apply(pd.to_numeric, errors='coerce').round(0).astype('float64')
fullWeatherData[['wind']]=fullWeatherData[['wind']].apply(pd.to_numeric, errors='coerce').round(1)

# Drop the 'snowfall' and 'precipitation' as well as the extra 'Peak sun hours/day_x'  columns as they are no longer needed after creating the annual averages
fullWeatherData = fullWeatherData.drop(['snowfall', 'precipitation', 'Peak sun hours/day_x'], axis=1)

# Rename the 'Peak sun hours/day_y' column to 'Peak sun hours/day'
fullWeatherData.rename(columns={'Peak sun hours/day_y': 'Peak sun hours/day'}, inplace=True)
fullWeatherData.head()

,zip,city,state,avg_relative_humidity_pct,temperature,wind,Peak sun hours/day,annual_avg_snowfall,annual_avg_precipitation
0,98253,Greenbank,WA,75.0,51.6,5.1,3.8,4.8,30.12
1,98271,Marysville,WA,78.0,50.7,6.4,3.8,4.8,30.12
2,98324,Carlsborg,WA,79.0,49.6,4.8,3.8,8.4,27.36
3,99148,Loon Lake,WA,67.0,47.1,8.0,3.8,45.6,20.76
4,98834,Methow,WA,59.0,51.0,2.1,3.8,45.6,14.52


In [ ]:
#-------------- Save the final cleaned dataframe to a CSV file --------------

fullWeatherData.to_csv(r"C:\Users\leedf\weatherData\finalCleanedWeatherDataByZip.csv", index=False)